**STEP 1: Set Up Environment**

In [1]:
# Display installation header with visual separator
print("="*70)
print(" INSTALLING PACKAGES")
print("="*70)
print("This will take 2-3 minutes. Please wait...")
print("(Dependency warnings are normal in Colab and can be ignored)\n")

# Install all required packages quietly with filtered output
# Packages installed:
# - langchain: Core LangChain framework
# - langchain-community: Community integrations for LangChain
# - langchain-google-genai: Google Gemini integration
# - sentence-transformers: For text embeddings
# - chromadb: Vector database for document storage
# - google-generativeai: Native Google Gemini SDK
# - google-search-results: For web search capabilities (optional)
# - pandas: Data manipulation and analysis
#
# The grep command filters out common dependency warnings to keep output clean
!pip install -q langchain langchain-community langchain-google-genai sentence-transformers chromadb google-generativeai google-search-results pandas 2>&1 | grep -v "dependency conflicts\|incompatible\|ERROR: pip's dependency" || true

# Display completion message
print("\n Installation complete!")
print("="*70)

 INSTALLING PACKAGES
This will take 2-3 minutes. Please wait...
(Dependency warnings are normal in Colab and can be ignored)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.8/20.8 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 83.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.3/132.3 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 5.5 MB/s eta 0:00:00
   ━━━

In [2]:
# Importing Libraries
# LangChain - Google Gemini Integration
from langchain_google_genai import ChatGoogleGenerativeAI

# LangChain - Core Components
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain, SequentialChain

# Utility Libraries
import os
from google.colab import userdata

In [3]:
#  API Key Configuration
# ============================================================================
# Securely retrieve and configure the Google Gemini API key from Colab Secrets.
# This approach keeps API keys secure and prevents accidental exposure in code.
# ============================================================================

# Retrieve API key from Colab Secrets and set as environment variable
# To add your API key:
# 1. Click the 🔑 icon in the left sidebar (Secrets)
# 2. Add a new secret named 'GEMINI_API_KEY'
# 3. Paste your Gemini API key as the value
# Get your free API key at: https://aistudio.google.com/app/api-keys?

os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_API_KEY")

print("✓ API Key configured successfully")
print("   (Key is stored securely and not displayed)")

✓ API Key configured successfully
   (Key is stored securely and not displayed)


**STEP 2: Initialize Language Model**

In [4]:
# ============================================================================
# Configure and initialize the Google Gemini model that will power our
# quiz generation. This model handles all AI-powered tasks including question
# generation, answer creation, and content analysis.
# ============================================================================

# Initialize Gemini LLM with optimized parameters

llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0.7,
    convert_system_message_to_human=True
)

print("✓ Gemini LLM initialized successfully")
print(f"   Model: gemini-2.0-flash")
print(f"   Temperature: 0.7 (balanced creativity)")

✓ Gemini LLM initialized successfully
   Model: gemini-2.0-flash
   Temperature: 0.7 (balanced creativity)


**STEP 3: Design Prompt Templates**

In [5]:
#  Chain 1 - Generate a question
# ============================================================================
# This chain creates a beginner-level question based on the provided topic.
# It's the first step in our sequential quiz generation pipeline.
# ============================================================================

# Define the prompt template for question generation
# This template takes a topic and generates an appropriate beginner question
question_prompt = PromptTemplate(
    input_variables=["topic"],
    template="Generate a beginner-level question related to the topic: {topic}"
)



In [6]:
#  Chain 2 - Answer the question
# ============================================================================
# This chain generates a clear, concise answer with explanation for the
# question created by Chain 1. It provides both the answer and reasoning.
# ============================================================================

# Define the prompt template for answer generation
# This template takes the generated question and produces an answer with explanation

answer_prompt = PromptTemplate(
    input_variables=["question"],
    template="Provide a clear answer with a short explanation for the question: {question}"
)

print("✓ Chain 2 (Answer Generation) prompt template created")
print("   Purpose: Generate clear answers with explanations")

✓ Chain 2 (Answer Generation) prompt template created
   Purpose: Generate clear answers with explanations


**STEP 4: Build Individual Chains**

In [7]:
#  Create individual chains
# ============================================================================
# Instantiate the individual chains that will be connected in a sequence.
# Each chain wraps the LLM with its specific prompt template and output key.
# ============================================================================

# Chain 1: Question Generation Chain
# Takes 'topic' as input, outputs 'question'

question_chain = LLMChain(llm=llm, prompt=question_prompt, output_key="question")
answer_chain = LLMChain(llm=llm, prompt=answer_prompt, output_key="answer")

print("✓ Individual chains created successfully")
print("   Chain 1: topic → question")
print("   Chain 2: question → answer")

✓ Individual chains created successfully
   Chain 1: topic → question
   Chain 2: question → answer


/tmp/ipython-input-3647612645.py:10: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  question_chain = LLMChain(llm=llm, prompt=question_prompt, output_key="question")


**STEP 5: Compose Sequential Chain**

In [8]:
# ============================================================================
# Connect the individual chains into a sequential workflow where the output
# of Chain 1 (question) automatically becomes the input for Chain 2 (answer).
# This creates an end-to-end quiz generation pipeline.
# ============================================================================

# Create the sequential chain that orchestrates the entire workflow


chain = SequentialChain(
    chains=[question_chain, answer_chain],
    input_variables=["topic"],
    output_variables=["question", "answer"],
    verbose=True
)

print("✓ Sequential chain created successfully")
print("   Pipeline: topic → [Chain 1] → question → [Chain 2] → answer")
print("   Verbose mode: ENABLED (will show execution details)")

✓ Sequential chain created successfully
   Pipeline: topic → [Chain 1] → question → [Chain 2] → answer
   Verbose mode: ENABLED (will show execution details)


**STEP 6: Run Quiz Generator**

In [9]:

# ============================================================================
# Run the complete sequential chain with user input and display the results.
# The pipeline will execute both chains automatically and return the final
# question-answer pair.
# ============================================================================

# Get user input for the quiz topic

topic = input("Enter a topic: ")

# Execute the sequential chain
# The chain will:
# 1. Generate a question from the topic (Chain 1)
# 2. Generate an answer for that question (Chain 2)
# 3. Return both outputs in a dictionary

response = chain.invoke({"topic": topic})


# ============================================================================
# DISPLAY RESULTS
# ============================================================================
# Present the generated quiz question and answer in a clean, readable format.
# ============================================================================

print("\n" + "="*70)
print(" GENERATED QUIZ")
print("="*70)

# Display the generated question
print("\n❓ Question:")
print(response["question"])

# Display the answer with explanation
print("\n Answer & Explanation:")
print(response["answer"])

print("\n" + "="*70)



Enter a topic: AI Agents


> Entering new SequentialChain chain...


/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:357: UserWarning: Convert_system_message_to_human will be deprecated!
  warnings.warn("Convert_system_message_to_human will be deprecated!")
/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:357: UserWarning: Convert_system_message_to_human will be deprecated!
  warnings.warn("Convert_system_message_to_human will be deprecated!")



> Finished chain.

 GENERATED QUIZ

❓ Question:
Okay, here's a beginner-level question about AI Agents:

**Question:** What is the main job of an AI Agent, and can you give a simple example of something an AI Agent might do?

 Answer & Explanation:
**Answer:** The main job of an AI Agent is to perceive its environment and take actions to achieve specific goals.

**Example:** An AI Agent in a self-driving car perceives the road, other cars, and traffic signals, and then takes actions like steering, accelerating, and braking to safely reach its destination.

